# 06 - Métodos de Gauss
Vamos aprender sobre como usar os métodos de Gauss para resolução de sistemas lineares quadráticos.

Crie uma nova branch (versão) do repositório:

```bash
git branch semana6
```

Faça o checkout nessa nova branch:

```bash
git checkout semana6
```

<hr />

## Atividade 1
Escreva os métodos de gauss dentro do arquivo $utils/algoritmos.py$.

In [ ]:
import numpy as np


def lu_pivot(A: np.ndarray):
    A = A.astype(float).copy()
    n = A.shape[0]
    P = np.eye(n)
    L = np.zeros((n, n))
    U = A.copy()

    for k in range(n):
        pivot = np.argmax(np.abs(U[k:, k])) + k
        if np.isclose(U[pivot, k], 0.0):
            raise np.linalg.LinAlgError("Matriz singular ou quase singular.")
        if pivot != k:
            U[[k, pivot], k:] = U[[pivot, k], k:]
            P[[k, pivot], :] = P[[pivot, k], :]
            if k > 0:
                L[[k, pivot], :k] = L[[pivot, k], :k]

        for i in range(k + 1, n):
            L[i, k] = U[i, k] / U[k, k]
            U[i, k:] -= L[i, k] * U[k, k:]

    np.fill_diagonal(L, 1.0)
    return P, L, U


def lb(L: np.ndarray, B: np.ndarray) -> np.ndarray:
    n = L.shape[0]
    Y = np.zeros(n)
    for i in range(n):
        Y[i] = B[i] - np.dot(L[i, :i], Y[:i])
    return Y


def uy(U: np.ndarray, Y: np.ndarray) -> np.ndarray:
    n = U.shape[0]
    X = np.zeros(n)
    for i in reversed(range(n)):
        if np.isclose(U[i, i], 0.0):
            raise np.linalg.LinAlgError(
                "U possui pivô nulo; sistema sem solução única."
            )
        X[i] = (Y[i] - np.dot(U[i, i + 1 :], X[i + 1 :])) / U[i, i]
    return X


def lu(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    P, L, U = lu_pivot(A)
    Pb = P @ B
    Y = lb(L, Pb)
    X = uy(U, Y)
    return X


def jacobi(A: np.ndarray, B: np.ndarray, k: int, TOL: float) -> np.ndarray:
    A = A.astype(float)
    B = B.astype(float)
    n = B.shape[0]
    X = np.zeros(n)
    Xk = np.zeros(n)

    D = np.diag(A)
    if np.any(D == 0):
        raise ValueError("A possui elementos diagonais nulos; Jacobi pode falhar.")

    R = A - np.diagflat(D)

    for _ in range(k):
        Xk = (B - R @ X) / D
        if np.linalg.norm(Xk - X, ord=2) < TOL:
            return Xk
        X = Xk.copy()
    return X


def seidel(A: np.ndarray, B: np.ndarray, k: int, TOL: float) -> np.ndarray:
    A = A.astype(float)
    B = B.astype(float)
    n = B.shape[0]
    X = np.zeros(n)

    for _ in range(k):
        Xk = X.copy()
        for i in range(n):
            s1 = np.dot(A[i, :i], X[:i])
            s2 = np.dot(A[i, i + 1 :], Xk[i + 1 :])
            X[i] = (B[i] - s1 - s2) / A[i, i]
        if np.linalg.norm(X - Xk, ord=2) < TOL:
            return X
    return X

Consideramos o seguinte sistema linear:

$$
\begin{cases}
x_1 + x_2 + x_3 = 1\\
4x_1 + 4x_2 + 2x_3 = 2\\
2x_1 + x_2 - x_3 = 0
\end{cases}
$$

Na sua forma matricial, este sistema é escrito como

$$
Ax = b \iff
\begin{bmatrix}
1 & 1 & 1 \\
4 & 4 & 2 \\
2 & 1 & -1
\end{bmatrix}
\begin{bmatrix}
x_1 \\
x_2 \\
x_3
\end{bmatrix}
=
\begin{bmatrix}
1 \\
2 \\
0
\end{bmatrix}
$$

Podemos resolver em Python usando a biblioteca NumPy.

In [2]:
import numpy as np

def main():
    A = np.array([
        [1, 1,  1],
        [4, 4,  2],
        [2, 1, -1]
    ], dtype=float)
    B = np.array([1, 2, 0], dtype=float)
    print("Matriz A:")
    print(A)
    print("\nVetor B:")
    print(B)
    print("\nSolução NumPy x:")
    X = np.linalg.solve(A, B)
    print(X)

if __name__ == "__main__":
    main()

Matriz A:
[[ 1.  1.  1.]
 [ 4.  4.  2.]
 [ 2.  1. -1.]]

Vetor B:
[1. 2. 0.]

Solução NumPy x:
[ 1. -1.  1.]


Agora, resolva usando os métodos de gauss e compare os resultados.

In [3]:
import sys
import os

# Sobe um nível para a raiz do projeto e adiciona ao caminho do sistema
sys.path.append(os.path.abspath(os.path.join('..')))

from utils.algoritmos import (
    lu,
    jacobi,
    seidel,
)

def main():
    A = np.array([[1, 1, 1], [4, 4, 2], [2, 1, -1]], dtype=float)
    B = np.array([1, 2, 0], dtype=float)
    print("Matriz A:")
    print(A)
    print("\nVetor B:")
    print(B)
    print("\nSolução NumPy x:")
    X = np.linalg.solve(A, B)
    print(X)
    print("\nSolução LU x:")
    X = lu(A, B)
    print(X)
    A = np.array([[2, 1, -1], [4, 4, 2], [1, 1, 1]], dtype=float)
    B = np.array([0, 2, 1], dtype=float)
    print("\nMatriz A:")
    print(A)
    print("\nVetor B:")
    print(B)
    print("\nSolução Jacobi x:")
    X = jacobi(A, B, 100, 1e-8)
    print(X)
    print("\nSolução Seidel x:")
    X = seidel(A, B, 100, 1e-8)
    print(X)


if __name__ == "__main__":
    main()

Matriz A:
[[ 1.  1.  1.]
 [ 4.  4.  2.]
 [ 2.  1. -1.]]

Vetor B:
[1. 2. 0.]

Solução NumPy x:
[ 1. -1.  1.]

Solução LU x:
[ 1. -1.  1.]

Matriz A:
[[ 2.  1. -1.]
 [ 4.  4.  2.]
 [ 1.  1.  1.]]

Vetor B:
[0. 2. 1.]

Solução Jacobi x:
[ 0.99999507 -0.99999329  0.99999799]

Solução Seidel x:
[ 1. -1.  1.]


## Atividade 2
Resolva o seguinte sistema pelo método de Jacobi e Gauss–Seidel:

$$
\begin{cases}
5x_1 + x_2 + x_3 = 50 \\
-x_1 + 3x_2 - x_3 = 10 \\
x_1 + 2x_2 + 10x_3 = -30
\end{cases}
$$

Use como critério de paragem tolerância inferior a $10^{-3}$  
e inicialize com $x^0 = y^0 = z^0 = 0$.

In [1]:
A = [
    [5.0, 1.0, 1.0],
    [-1.0, 3.0, -1.0],
    [1.0, 2.0, 10.0],
]
b = [50.0, 10.0, -30.0]
x_initial = [0.0, 0.0, 0.0]
tolerance = 1e-3
max_iterations = 1000


def infinity_norm_difference(first, second):
    return max(abs(first[index] - second[index]) for index in range(len(first)))


def jacobi(matrix, right_hand_side, initial, tolerance, max_iterations):
    current = initial.copy()
    for iteration in range(1, max_iterations + 1):
        next_value = []
        for row in range(len(right_hand_side)):
            off_diagonal_sum = sum(
                matrix[row][column] * current[column]
                for column in range(len(right_hand_side))
                if column != row
            )
            next_value.append(
                (right_hand_side[row] - off_diagonal_sum) / matrix[row][row]
            )

        error = infinity_norm_difference(next_value, current)
        if error < tolerance:
            return next_value, iteration, error
        current = next_value

    raise RuntimeError("Jacobi não convergiu dentro do limite de iterações.")


def gauss_seidel(matrix, right_hand_side, initial, tolerance, max_iterations):
    current = initial.copy()
    for iteration in range(1, max_iterations + 1):
        previous = current.copy()
        for row in range(len(right_hand_side)):
            lower_sum = sum(
                matrix[row][column] * current[column]
                for column in range(row)
            )
            upper_sum = sum(
                matrix[row][column] * previous[column]
                for column in range(row + 1, len(right_hand_side))
            )
            current[row] = (
                right_hand_side[row] - lower_sum - upper_sum
            ) / matrix[row][row]

        error = infinity_norm_difference(current, previous)
        if error < tolerance:
            return current, iteration, error

    raise RuntimeError("Gauss-Seidel não convergiu dentro do limite de iterações.")


def residual_infinity_norm(matrix, solution, right_hand_side):
    residuals = [
        sum(matrix[row][column] * solution[column] for column in range(len(solution)))
        - right_hand_side[row]
        for row in range(len(solution))
    ]
    return max(abs(value) for value in residuals)


for method_name, method in (("Jacobi", jacobi), ("Gauss-Seidel", gauss_seidel)):
    solution, iterations, final_error = method(
        A, b, x_initial, tolerance, max_iterations
    )
    residual = residual_infinity_norm(A, solution, b)
    assert final_error < tolerance
    assert residual < 1e-2
    print(f"{method_name}:")
    print(f"  solução (x1, x2, x3) = ({solution[0]:.6f}, {solution[1]:.6f}, {solution[2]:.6f})")
    print(f"  iterações = {iterations}; diferença final = {final_error:.3e}")
    print(f"  norma do resíduo = {residual:.3e}")

Jacobi:
  solução (x1, x2, x3) = (9.999844, 5.000123, -5.000130)
  iterações = 10; diferença final = 4.480e-04
  norma do resíduo = 1.212e-03
Gauss-Seidel:
  solução (x1, x2, x3) = (9.999908, 4.999905, -4.999972)
  iterações = 6; diferença final = 7.467e-04
  norma do resíduo = 5.259e-04


## Atividade 3
Faça uma permutação de linhas no sistema abaixo e resolva pelos métodos de Jacobi e Gauss–Seidel:

$$
\begin{cases}
x_{1} + 10x_{2} + 3x_{3} = 27 \\
4x_{1} + x_{3} = 6 \\
2x_{1} + x_{2} + 4x_{3} = 12
\end{cases}
$$

In [2]:
matrix_original = [
    [1.0, 10.0, 3.0],
    [4.0, 0.0, 1.0],
    [2.0, 1.0, 4.0],
]
right_hand_side_original = [27.0, 6.0, 12.0]
row_order = [1, 0, 2]

matrix_permuted = [matrix_original[row] for row in row_order]
right_hand_side_permuted = [right_hand_side_original[row] for row in row_order]
initial_guess = [0.0, 0.0, 0.0]
tolerance = 1e-8
max_iterations = 1000

assert all(
    abs(matrix_permuted[row][row])
    > sum(abs(value) for column, value in enumerate(matrix_permuted[row]) if column != row)
    for row in range(3)
)


def residual_norm(matrix, solution, right_hand_side):
    residuals = [
        sum(matrix[row][column] * solution[column] for column in range(len(solution)))
        - right_hand_side[row]
        for row in range(len(solution))
    ]
    return max(abs(value) for value in residuals)


print("Ordem das equações: 2, 1, 3")
print("Matriz permutada:")
for row in matrix_permuted:
    print(row)

for method_name, method in (("Jacobi", jacobi), ("Gauss-Seidel", gauss_seidel)):
    solution, iterations, final_error = method(
        matrix_permuted,
        right_hand_side_permuted,
        initial_guess,
        tolerance,
        max_iterations,
    )
    residual = residual_norm(matrix_original, solution, right_hand_side_original)
    assert final_error < tolerance
    assert residual < 1e-7
    print(f"{method_name}: x = ({solution[0]:.8f}, {solution[1]:.8f}, {solution[2]:.8f})")
    print(f"  iterações = {iterations}; erro final = {final_error:.3e}; resíduo = {residual:.3e}")

Ordem das equações: 2, 1, 3
Matriz permutada:
[4.0, 0.0, 1.0]
[1.0, 10.0, 3.0]
[2.0, 1.0, 4.0]
Jacobi: x = (1.00000000, 2.00000000, 2.00000000)
  iterações = 27; erro final = 5.801e-09; resíduo = 2.056e-08
Gauss-Seidel: x = (1.00000000, 2.00000000, 2.00000000)
  iterações = 13; erro final = 6.405e-09; resíduo = 1.354e-08


## Atividade 4
O circuito linear da Figura pode ser modelado pelo sistema dado a seguir.  
Escreva esse sistema na forma matricial sendo as tensões $V_1, V_2, V_3, V_4, V_5$ as cinco incógnitas.  
Resolva esse problema quando $V = 127$ e:

- $R_1 = R_2 = R_3 = R_4 = 2$, $R_5 = R_6 = R_7 = 100$ e $R_8 = 50$
- $R_1 = R_2 = R_3 = R_4 = 2$, $R_5 = 50$, $R_6 = R_7 = R_8 = 100$

Sistema de equações:

$$
V_1 = V
$$

$$
\frac{V_1 - V_2}{R_1} + \frac{V_5 - V_2}{R_2} - \frac{V_2}{R_5} = 0
$$

$$
\frac{V_2 - V_3}{R_2} + \frac{V_4 - V_3}{R_3} - \frac{V_3}{R_6} = 0
$$

$$
\frac{V_3 - V_4}{R_3} + \frac{V_5 - V_4}{R_4} - \frac{V_4}{R_7} = 0
$$

$$
\frac{V_4 - V_5}{R_4} - \frac{V_5}{R_8} = 0
$$

![Circuito](https://www.ufrgs.br/reamat/CalculoNumerico/livro-py/main11x.png)

Complete a tabela abaixo representando a solução com **4 algarismos significativos**:

| Caso | $V_1$ | $V_2$ | $V_3$ | $V_4$ | $V_5$ |
|------|-------|-------|-------|-------|-------|
| a    |       |       |       |       |       |
| b    |       |       |       |       |       |

Então, refaça este problema reduzindo o sistema para apenas 4 incógnitas ($V_2, V_3, V_4, V_5$).

In [ ]:
import math
import numpy as np


SOURCE_VOLTAGE = 127.0


def build_circuit_system(resistances):
    resistance_1, resistance_2, resistance_3, resistance_4 = resistances[:4]
    resistance_5, resistance_6, resistance_7, resistance_8 = resistances[4:]

    matrix = np.array(
        [
            [1.0, 0.0, 0.0, 0.0, 0.0],
            [
                -1.0 / resistance_1,
                1.0 / resistance_1 + 1.0 / resistance_2 + 1.0 / resistance_5,
                0.0,
                0.0,
                -1.0 / resistance_2,
            ],
            [
                0.0,
                -1.0 / resistance_2,
                1.0 / resistance_2 + 1.0 / resistance_3 + 1.0 / resistance_6,
                -1.0 / resistance_3,""
                0.0,
            ],
            [
                0.0,
                0.0,
                -1.0 / resistance_3,
                1.0 / resistance_3 + 1.0 / resistance_4 + 1.0 / resistance_7,
                -1.0 / resistance_4,
            ],
            [
                0.0,
                0.0,
                0.0,
                -1.0 / resistance_4,
                1.0 / resistance_4 + 1.0 / resistance_8,
            ],
        ],
        dtype=float,
    )
    right_hand_side = np.array([SOURCE_VOLTAGE, 0.0, 0.0, 0.0, 0.0])
    return matrix, right_hand_side


def format_significant(value: float, digits: int = 4) -> str:
    if value == 0.0:
        return f"{value:.{digits - 1}f}"
    decimal_places = digits - 1 - math.floor(math.log10(abs(value)))
    return f"{value:.{max(decimal_places, 0)}f}"


cases = {
    "a": (2.0, 2.0, 2.0, 2.0, 100.0, 100.0, 100.0, 50.0),
    "b": (2.0, 2.0, 2.0, 2.0, 50.0, 100.0, 100.0, 100.0),
}

case_solutions = {}
for case_name, resistances in cases.items():
    matrix, right_hand_side = build_circuit_system(resistances)
    full_solution = np.linalg.solve(matrix, right_hand_side)

    reduced_matrix = matrix[1:, 1:]
    reduced_right_hand_side = right_hand_side[1:].copy()
    reduced_right_hand_side[0] = SOURCE_VOLTAGE / resistances[0]
    reduced_solution = np.concatenate(
        ([SOURCE_VOLTAGE], np.linalg.solve(reduced_matrix, reduced_right_hand_side))
    )

    assert np.allclose(full_solution, reduced_solution, atol=1e-10)
    assert np.linalg.norm(matrix @ full_solution - right_hand_side, ord=np.inf) < 1e-10
    case_solutions[case_name] = full_solution

    print(f"Caso {case_name}: R = {resistances} Ω")
    print("Matriz A (sistema com 5 incógnitas):")
    print(np.array2string(matrix, precision=6, suppress_small=True))
    print("Vetor b:", right_hand_side)
    print("Matriz reduzida (V1 = 127 V):")
    print(np.array2string(reduced_matrix, precision=6, suppress_small=True))
    print("Vetor reduzido:", reduced_right_hand_side)
    print("Solução (sistema completo e reduzido):")
    print(" ".join(
        f"V{index + 1}={format_significant(value)} V"
        for index, value in enumerate(full_solution)
    ))
    print()

print("Tabela final (4 algarismos significativos):")
print("Caso | V1 (V) | V2 (V) | V3 (V) | V4 (V) | V5 (V)")
for case_name, solution in case_solutions.items():
    formatted_values = " | ".join(format_significant(value) for value in solution)
    print(f"{case_name} | {formatted_values}")

Caso a: R = (2.0, 2.0, 2.0, 2.0, 100.0, 100.0, 100.0, 50.0) Ω
Matriz A (sistema com 5 incógnitas):
[[ 1.    0.    0.    0.    0.  ]
 [-0.5   1.01  0.    0.   -0.5 ]
 [ 0.   -0.5   1.01 -0.5   0.  ]
 [ 0.    0.   -0.5   1.01 -0.5 ]
 [ 0.    0.    0.   -0.5   0.52]]
Vetor b: [127.   0.   0.   0.   0.]
Matriz reduzida (V1 = 127 V):
[[ 1.01  0.    0.   -0.5 ]
 [-0.5   1.01 -0.5   0.  ]
 [ 0.   -0.5   1.01 -0.5 ]
 [ 0.    0.   -0.5   0.52]]
Vetor reduzido: [63.5  0.   0.   0. ]
Solução (sistema completo e reduzido):
V1=127.0 V V2=108.1 V V3=100.5 V V4=94.96 V V5=91.31 V

Caso b: R = (2.0, 2.0, 2.0, 2.0, 50.0, 100.0, 100.0, 100.0) Ω
Matriz A (sistema com 5 incógnitas):
[[ 1.    0.    0.    0.    0.  ]
 [-0.5   1.02  0.    0.   -0.5 ]
 [ 0.   -0.5   1.01 -0.5   0.  ]
 [ 0.    0.   -0.5   1.01 -0.5 ]
 [ 0.    0.    0.   -0.5   0.51]]
Vetor b: [127.   0.   0.   0.   0.]
Matriz reduzida (V1 = 127 V):
[[ 1.02  0.    0.   -0.5 ]
 [-0.5   1.01 -0.5   0.  ]
 [ 0.   -0.5   1.01 -0.5 ]
 [ 0.    0.   -

## Versionando o código

Submeta a branch para o servidor:

```bash
git add .
git commit -m "Semana 6"
git push origin semana6
```